# Filtro de linha regional (etapa 6.3)

Implementa o [ADR 0018](../../docs/adr/0018-filtro-de-linha-regional.md): o **gestor** vê só os pedidos e os clientes das **UFs da sua região**; analista, DPO e engenharia veem todas as linhas.

| Tabela da gold | Coluna usada pelo filtro | Função |
|---|---|---|
| `dim_customer` | `customer_state` | `ops.filtro_regiao_uf` |
| `fact_orders` | `customer_id` (a UF vem do cadastro em `silver.customers`) | `ops.filtro_regiao_cliente` |

O recorte fica na tabela `ops.acesso_regional` (grupo, UF). Mudar a região do gestor é mudar linhas dessa tabela, sem tocar nas funções.

**Parâmetros**

| Widget | Valores | Para que serve |
|---|---|---|
| `modo` | `simulacao` (padrão), `executar`, `reverter`, `testar_perfil` | o que o notebook faz |
| `ufs_gestor` | lista de UFs separadas por vírgula (padrão `SP,RJ,MG,ES`, a região Sudeste) | recorte do gestor |
| `fonte_uf` | `silver` (padrão) ou `copia_ops` | de onde o filtro de `fact_orders` lê a UF do cliente; use `copia_ops` só se o gestor receber erro de permissão em `silver.customers` |

> **Cuidado:** a atualização do pipeline da gold deve rodar como `olist_engenharia` (isenta do filtro). Fora desse grupo o pipeline leria só as linhas do recorte.

In [ ]:
# O QUE FAZ: define parâmetros, grupos e as tabelas que recebem o filtro.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (configuração).
#
# FUNÇÕES NATIVAS E SQL:
# - dbutils.widgets.dropdown / text / get: parâmetros no topo do notebook.
#
# REGRAS:
# - ALVOS: tabela da gold -> (coluna lida pelo filtro, função de filtro). Um filtro por tabela (limite do Unity Catalog).
# - UFS_GESTOR: lista limpa e em maiúsculas; só aceita siglas de 2 letras.
from pyspark.sql import functions as F

CATALOG = "olist"
dbutils.widgets.dropdown("modo", "simulacao", ["simulacao", "executar", "reverter", "testar_perfil"], "Modo")
dbutils.widgets.text("ufs_gestor", "SP,RJ,MG,ES", "UFs do gestor")
dbutils.widgets.dropdown("fonte_uf", "silver", ["silver", "copia_ops"], "Fonte da UF do cliente")
MODO = dbutils.widgets.get("modo")
FONTE_UF = dbutils.widgets.get("fonte_uf")
UFS_GESTOR = sorted({u.strip().upper() for u in dbutils.widgets.get("ufs_gestor").split(",") if u.strip()})
assert UFS_GESTOR and all(len(u) == 2 and u.isalpha() for u in UFS_GESTOR), f"UFs inválidas: {UFS_GESTOR}"

GRUPOS = {"analista": "olist_analista", "gestor": "olist_gestor", "dpo": "olist_dpo", "engenharia": "olist_engenharia"}
ALVOS = {
    "dim_customer": ("customer_state", "filtro_regiao_uf"),
    "fact_orders": ("customer_id", "filtro_regiao_cliente"),
}
REGRAS = f"{CATALOG}.ops.acesso_regional"
FALHAS = []
print("modo:", MODO, "| UFs do gestor:", ", ".join(UFS_GESTOR), "| fonte da UF:", FONTE_UF)

In [ ]:
# O QUE FAZ: prepara a tabela de verificações (a mesma da etapa 6.2) e a função que grava resultados.
#
# FUNÇÕES DECLARADAS:
# - registrar(etapa, linhas, apagar_antes): apaga as linhas anteriores da etapa (ou dos perfis) e grava as novas em
#   quality.validacao_governanca. Rodar de novo troca o resultado, não duplica.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: cria a tabela de resultados uma vez, já com comentário em tabela e colunas.
# - DELETE FROM t WHERE ...: remove as linhas antigas (Delta). spark.createDataFrame(linhas, "schema DDL"): cria o
#   DataFrame a partir de tuplas. F.lit(x): coluna com valor fixo. F.current_timestamp(): hora da execução.
# - df.write.mode("append").saveAsTable(t): acrescenta as linhas à tabela.

VALIDACAO = f"{CATALOG}.quality.validacao_governanca"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDACAO} (
  etapa STRING COMMENT 'Etapa da Fase 6 que gerou a verificação (6.2-grants, 6.2-mascaras, 6.2-perfil, ...).',
  perfil STRING COMMENT 'Perfil avaliado (analista, gestor, dpo, engenharia) ou geral quando vale para todos.',
  verificacao STRING COMMENT 'O que foi verificado.',
  esperado STRING COMMENT 'Valor esperado pela regra.',
  obtido STRING COMMENT 'Valor encontrado no ambiente.',
  resultado STRING COMMENT 'OK, REPROVADA ou REGISTRO (só guarda um valor de referência).',
  executado_em TIMESTAMP COMMENT 'Momento da execução.'
)
COMMENT 'Resultado das verificações de governança e LGPD (grants, máscaras, filtros de linha, exclusão de cliente). Nenhuma linha guarda dado pessoal.'
""")


def registrar(etapa: str, linhas: list, apagar_antes: str = "etapa") -> None:
    """Grava as verificações. linhas = [(perfil, verificacao, esperado, obtido, resultado)]."""
    if apagar_antes == "etapa":
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}'")
    else:
        perfis = sorted({l[0] for l in linhas})
        lista = ", ".join(f"'{p}'" for p in perfis)
        spark.sql(f"DELETE FROM {VALIDACAO} WHERE etapa = '{etapa}' AND perfil IN ({lista})")
    df = (spark.createDataFrame(linhas, "perfil string, verificacao string, esperado string, obtido string, resultado string")
          .withColumn("etapa", F.lit(etapa))
          .withColumn("executado_em", F.current_timestamp())
          .select("etapa", "perfil", "verificacao", "esperado", "obtido", "resultado", "executado_em"))
    df.write.mode("append").saveAsTable(VALIDACAO)

In [ ]:
# O QUE FAZ: confere os pré-requisitos e mostra o plano, sem alterar nada.
#
# FUNÇÕES DECLARADAS:
# - existe(nome): verdadeiro se a tabela ou a view existe no catálogo.
#
# FUNÇÕES NATIVAS E SQL:
# - spark.catalog.tableExists: confere a existência de uma tabela pelo nome completo.
# - SHOW GROUPS: lista os grupos de conta (os grants e os filtros só enxergam esses).
def existe(nome: str) -> bool:
    return spark.catalog.tableExists(nome)


grupos_conta = {r[0] for r in spark.sql("SHOW GROUPS").collect()}
faltam = [g for g in GRUPOS.values() if g not in grupos_conta]
assert not faltam, f"Grupos ausentes: {faltam}. Rode antes a etapa 6.2."
for t in ALVOS:
    assert existe(f"{CATALOG}.gold.{t}"), f"{CATALOG}.gold.{t} não existe"
assert existe(f"{CATALOG}.silver.customers"), "silver.customers não existe"

print("Plano:")
print(f"  1. {REGRAS}: grupo {GRUPOS['gestor']} -> {', '.join(UFS_GESTOR)}")
if FONTE_UF == "copia_ops":
    print(f"  2. {CATALOG}.ops.cliente_uf: cópia de customer_id e UF, usada pelo filtro de fact_orders")
for t, (col, fn) in ALVOS.items():
    print(f"  - {CATALOG}.gold.{t}: filtro ops.{fn} sobre {col}")
print("Isentos (veem todas as linhas): engenharia, dpo e analista. Sem grupo do projeto: nenhuma linha.")

In [ ]:
# O QUE FAZ: cria a tabela de regras e as duas funções de filtro; aplica ou remove o filtro conforme o modo.
#
# FUNÇÕES DECLARADAS:
# - verbo(tabela): ALTER MATERIALIZED VIEW ou ALTER TABLE, conforme o tipo do objeto.
# - tentar(descricao, principal, alternativa): roda o comando; se falhar tenta a alternativa; guarda a falha em FALHAS.
#
# FUNÇÕES NATIVAS E SQL:
# - CREATE TABLE IF NOT EXISTS ... COMMENT: tabela de regras (grupo, uf) documentada.
# - DELETE + INSERT: recarga idempotente das UFs do gestor (rodar de novo não duplica).
# - CREATE OR REPLACE FUNCTION ... RETURNS BOOLEAN: função de filtro; linha com resultado true é mostrada.
# - is_account_group_member('grupo'): verdadeiro se quem consulta está no grupo de conta.
# - EXISTS (SELECT ...): consulta a tabela de regras dentro da função, para o recorte ser dado e não código.
# - ALTER ... SET ROW FILTER fn ON (coluna): liga o filtro; ALTER ... DROP ROW FILTER: remove.
# - information_schema.tables.table_type: distingue MATERIALIZED_VIEW de tabela comum.
ENG, DPO, GES, ANA = (GRUPOS[k] for k in ("engenharia", "dpo", "gestor", "analista"))


def verbo(t: str) -> str:
    tp = spark.sql(f"SELECT table_type FROM {CATALOG}.information_schema.tables "
                   f"WHERE table_schema = 'gold' AND table_name = '{t}'").first()[0]
    return "MATERIALIZED VIEW" if tp == "MATERIALIZED_VIEW" else "TABLE"


def tentar(descricao, principal, alternativa=None):
    for sql in [x for x in (principal, alternativa) if x]:
        try:
            spark.sql(sql)
            print("ok   ", descricao)
            return True
        except Exception as e:
            ultimo = str(e)[:250]
    FALHAS.append((descricao, ultimo))
    print("FALHA", descricao, "->", ultimo[:160])
    return False


if MODO == "executar":
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {REGRAS} (
      grupo STRING COMMENT 'Grupo de conta que recebe o recorte.',
      uf STRING COMMENT 'Sigla da UF que o grupo pode ver.')
      COMMENT 'Recorte regional do filtro de linha (ADR 0018): grupo -> UFs visíveis. Mudar o recorte é mudar linhas aqui.'""")
    spark.sql(f"DELETE FROM {REGRAS} WHERE grupo = '{GES}'")
    valores = ", ".join(f"('{GES}', '{u}')" for u in UFS_GESTOR)
    spark.sql(f"INSERT INTO {REGRAS} VALUES {valores}")
    print("ok    regras:", ", ".join(UFS_GESTOR))

    isento = f"is_account_group_member('{ENG}') OR is_account_group_member('{DPO}') OR is_account_group_member('{ANA}')"
    spark.sql(f"""CREATE OR REPLACE FUNCTION {CATALOG}.ops.filtro_regiao_uf(p_uf STRING) RETURNS BOOLEAN
      COMMENT 'Filtro de linha por UF: engenharia, dpo e analista veem tudo; o gestor vê só as UFs de ops.acesso_regional.'
      RETURN {isento} OR (is_account_group_member('{GES}')
        AND EXISTS (SELECT 1 FROM {REGRAS} r WHERE r.grupo = '{GES}' AND r.uf = p_uf))""")

    if FONTE_UF == "copia_ops":
        spark.sql(f"CREATE OR REPLACE TABLE {CATALOG}.ops.cliente_uf "
                  f"COMMENT 'Cópia de customer_id e UF usada pelo filtro de fact_orders (só quando fonte_uf = copia_ops).' "
                  f"AS SELECT customer_id, customer_state AS uf FROM {CATALOG}.silver.customers")
        origem = f"{CATALOG}.ops.cliente_uf c"
        coluna_uf = "c.uf"
    else:
        origem = f"{CATALOG}.silver.customers c"
        coluna_uf = "c.customer_state"
    spark.sql(f"""CREATE OR REPLACE FUNCTION {CATALOG}.ops.filtro_regiao_cliente(p_customer_id STRING) RETURNS BOOLEAN
      COMMENT 'Filtro de linha de fact_orders: a UF do pedido vem do cadastro do cliente; o gestor vê só as UFs de ops.acesso_regional.'
      RETURN {isento} OR (is_account_group_member('{GES}')
        AND EXISTS (SELECT 1 FROM {origem} JOIN {REGRAS} r ON r.uf = {coluna_uf}
                    WHERE c.customer_id = p_customer_id AND r.grupo = '{GES}'))""")
    print("ok    funções criadas")

    for t, (col, fn) in ALVOS.items():
        alvo = f"{CATALOG}.gold.{t}"
        tentar(f"filtro {t}", f"ALTER {verbo(t)} {alvo} SET ROW FILTER {CATALOG}.ops.{fn} ON ({col})",
               f"ALTER TABLE {alvo} SET ROW FILTER {CATALOG}.ops.{fn} ON ({col})")
elif MODO == "reverter":
    for t in ALVOS:
        alvo = f"{CATALOG}.gold.{t}"
        tentar(f"remover filtro {t}", f"ALTER {verbo(t)} {alvo} DROP ROW FILTER", f"ALTER TABLE {alvo} DROP ROW FILTER")
else:
    print(f"Modo {MODO}: nada alterado nesta célula.")

if FALHAS:
    print("\nFalhas:")
    for d, e in FALHAS:
        print(" -", d, "->", e)

In [ ]:
# O QUE FAZ: modo `executar`. Confere no catálogo se os filtros estão ligados e se as regras e as funções existem; grava a etapa 6.3-filtros.
#
# FUNÇÕES DECLARADAS:
# - nenhuma (usa registrar).
#
# FUNÇÕES NATIVAS E SQL:
# - information_schema.row_filters: um registro por tabela com filtro (nome da função e coluna).
# - information_schema.routines: lista as funções do schema ops.
# - spark.table(...).collect(): lê poucas linhas (catálogo e regras), nunca dados grandes.
if MODO == "executar":
    rf = spark.sql(f"SELECT * FROM {CATALOG}.information_schema.row_filters").collect()
    # colunas lidas por posição: catálogo, schema, tabela, nome do filtro, colunas-alvo
    ligados = {r[2]: str(r[3]) for r in rf if r[1] == "gold"}
    linhas = []
    for t, (col, fn) in ALVOS.items():
        obtido = ligados.get(t, "sem filtro")
        linhas.append(("geral", f"filtro de linha ligado em gold.{t}", f"{CATALOG}.ops.{fn}", obtido,
                       "OK" if obtido.endswith(fn) else "REPROVADA"))
    ufs = sorted(r[0] for r in spark.table(REGRAS).where(F.col("grupo") == GRUPOS["gestor"]).select("uf").collect())
    linhas.append(("gestor", "UFs do recorte em ops.acesso_regional", ", ".join(UFS_GESTOR), ", ".join(ufs),
                   "OK" if ufs == UFS_GESTOR else "REPROVADA"))
    fns = {r[0] for r in spark.sql(f"SELECT routine_name FROM {CATALOG}.information_schema.routines "
                                   f"WHERE routine_schema = 'ops'").collect()}
    esperadas = {"filtro_regiao_uf", "filtro_regiao_cliente"}
    linhas.append(("geral", "funções de filtro existem em ops", ", ".join(sorted(esperadas)), ", ".join(sorted(esperadas & fns)),
                   "OK" if esperadas <= fns else "REPROVADA"))
    registrar("6.3-filtros", linhas)
    for l in linhas:
        print(f"[{l[4]:9}] {l[0]:10} {l[1]} | obtido: {l[3]}")
    if any(l[4] != "OK" for l in linhas) or FALHAS:
        raise RuntimeError("Verificação reprovada ou comando com falha; veja acima e a tabela quality.validacao_governanca")
else:
    print("Rode com modo = executar para verificar.")

In [ ]:
# O QUE FAZ: modo `testar_perfil`. Detecta o seu perfil, conta o que você enxerga em fact_orders e dim_customer e grava a etapa 6.3-perfil.
#
# FUNÇÕES DECLARADAS:
# - membro(grupo): verdadeiro se o seu usuário está no grupo de conta.
# - lido(chave): lê da tabela de verificações o valor de referência gravado por um perfil sem restrição.
#
# FUNÇÕES NATIVAS E SQL:
# - is_account_group_member('grupo'): mesma função usada nos filtros.
# - count, countDistinct, join e isin: contam as linhas e as UFs visíveis.
# - spark.table(...).where(...).first(): lê uma linha de referência.
#
# FLUXO:
# - Perfil sem restrição (engenharia, dpo ou analista): grava as referências (total de pedidos, de clientes e pedidos do recorte do gestor).
# - Gestor: compara com as referências; deve ver só as UFs do recorte e o mesmo número de pedidos do recorte.
def membro(grupo: str) -> bool:
    try:
        return bool(spark.sql(f"SELECT is_account_group_member('{grupo}')").first()[0])
    except Exception:
        return False


def lido(chave: str):
    r = (spark.table(VALIDACAO).where((F.col("etapa") == "6.3-perfil") & (F.col("verificacao") == chave)
         & F.col("perfil").isin("engenharia", "dpo")).orderBy(F.col("executado_em").desc()).first())
    return None if r is None else int(r["obtido"])


if MODO == "testar_perfil":
    membros = {p: membro(g) for p, g in GRUPOS.items()}
    perfil = next((p for p in ("engenharia", "dpo", "gestor", "analista") if membros[p]), "nenhum")
    efetivo = "analista" if perfil == "nenhum" else perfil
    print("Grupos do projeto a que você pertence:", [GRUPOS[p] for p, v in membros.items() if v] or "nenhum")
    print("Perfil considerado:", efetivo)

    fo = spark.table(f"{CATALOG}.gold.fact_orders")
    dc = spark.table(f"{CATALOG}.gold.dim_customer")
    pedidos = fo.count()
    clientes = dc.select("customer_id").distinct().count()
    ufs_vistas = sorted(r[0] for r in dc.select("customer_state").distinct().collect())
    ref_p, ref_c, ref_g = lido("referencia_pedidos_total"), lido("referencia_clientes_total"), lido("referencia_pedidos_do_recorte_do_gestor")
    linhas = []
    if efetivo in ("engenharia", "dpo"):
        cli = spark.table(f"{CATALOG}.silver.customers").select("customer_id", F.col("customer_state").alias("uf"))
        do_recorte = fo.join(cli, "customer_id").where(F.col("uf").isin(UFS_GESTOR)).count()
        linhas += [(efetivo, "referencia_pedidos_total", "", str(pedidos), "REGISTRO"),
                   (efetivo, "referencia_clientes_total", "", str(clientes), "REGISTRO"),
                   (efetivo, "referencia_pedidos_do_recorte_do_gestor", "", str(do_recorte), "REGISTRO"),
                   (efetivo, "vê todas as UFs em dim_customer", "27", str(len(ufs_vistas)), "OK" if len(ufs_vistas) == 27 else "REPROVADA")]
    elif efetivo == "analista":
        assert ref_p is not None, "Rode antes este modo como engenharia ou dpo, para gravar as referências."
        linhas += [(efetivo, "pedidos visíveis = total de referência", str(ref_p), str(pedidos), "OK" if pedidos == ref_p else "REPROVADA"),
                   (efetivo, "clientes visíveis = total de referência", str(ref_c), str(clientes), "OK" if clientes == ref_c else "REPROVADA"),
                   (efetivo, "vê todas as UFs em dim_customer", "27", str(len(ufs_vistas)), "OK" if len(ufs_vistas) == 27 else "REPROVADA")]
    else:
        assert ref_p is not None and ref_g is not None, "Rode antes este modo como engenharia ou dpo, para gravar as referências."
        linhas += [("gestor", "UFs visíveis dentro do recorte", ", ".join(UFS_GESTOR), ", ".join(ufs_vistas),
                    "OK" if ufs_vistas and set(ufs_vistas) <= set(UFS_GESTOR) else "REPROVADA"),
                   ("gestor", "pedidos visíveis = pedidos do recorte", str(ref_g), str(pedidos), "OK" if pedidos == ref_g else "REPROVADA"),
                   ("gestor", "clientes visíveis menos que o total", f"0 < n < {ref_c}", str(clientes), "OK" if 0 < clientes < ref_c else "REPROVADA")]
    registrar("6.3-perfil", linhas, apagar_antes="perfil")
    for l in linhas:
        print(f"[{l[4]:9}] {l[0]:10} {l[1]} | esperado: {l[2] or '-'} | obtido: {l[3]}")
    print("\nO que você vê em fact_orders:")
    fo.groupBy().agg(F.count("*").alias("pedidos"), F.countDistinct("customer_id").alias("clientes")).show()
else:
    print("Rode com modo = testar_perfil.")